# minecu_ref T4 baseline

Measures the hand-written CUDA from `Infatoshi/mine.cu` on this T4.
Everything runs in this notebook; the only host input is the two
base64 payloads, which are hash-checked before anything is built.
Self-verifying: the first three battery entries are correctness
gates, and a failing gate makes the run exit non-zero.

## What this produces

1. `SUMMARY.txt` -- the full log, which is the primary artefact.
2. `log_<entry>.txt` -- one per battery entry.
3. `kernels_sm75.sass` -- the SASS, so the run carries its own
   evidence that the binary is sm_75 with no sm_80+ instructions.
4. `minecu_bench_sm_75` -- the binary, so a reader can rebuild.

No number in this notebook is transcribed from the mine.cu README.
Every one is produced by the instrument in this notebook.

In [ ]:

import base64, hashlib, json, os, re, subprocess, sys, textwrap, time

BUDGET_S = 1500
OUT = "/kaggle/working"
os.makedirs(OUT, exist_ok=True)

EXPECT = {"c6ed8f06ee64d40a8a8ab7ebce3b3ceea1e3ac93f9ee50cfc1e63d3f090d878f": "src/kernels.cu", "990fdfc069ff8e59dcce5ee85ec2068a99b7b575b5800b1429185b066d3964a5": "bench/bench.cu"}
PAYLOADS = [["kernels.cu", "..."], ["bench.cu", "..."]]
BATTERY = [["selftest_w16", ["--batch", "4096", "--world", "16", "--res", "32", "--selftest-only"]], ["selftest_w32", ["--batch", "4096", "--world", "32", "--res", "32", "--selftest-only"]], ["selftest_w48", ["--batch", "4096", "--world", "48", "--res", "32", "--selftest-only"]], ["headline_w16", ["--batch", "32768", "--world", "16", "--res", "32", "--max-steps", "32", "--iters", "200", "--graph-iters", "200"]], ["sweep_w16", ["--batch", "32768", "--world", "16", "--res", "32", "--max-steps", "32", "--iters", "60", "--graph-iters", "60"]], ["batch4k_w16", ["--batch", "4096", "--world", "16", "--res", "32", "--max-steps", "32", "--iters", "100", "--graph-iters", "100"]], ["batch8k_w16", ["--batch", "8192", "--world", "16", "--res", "32", "--max-steps", "32", "--iters", "100", "--graph-iters", "100"]], ["batch16k_w16", ["--batch", "16384", "--world", "16", "--res", "32", "--max-steps", "32", "--iters", "100", "--graph-iters", "100"]], ["w32_32px", ["--batch", "32768", "--world", "32", "--res", "32", "--max-steps", "32", "--iters", "60", "--graph-iters", "60"]], ["w16_64px", ["--batch", "32768", "--world", "16", "--res", "64", "--max-steps", "32", "--iters", "30", "--graph-iters", "30"]], ["w16_ms8", ["--batch", "32768", "--world", "16", "--res", "32", "--max-steps", "8", "--iters", "60", "--graph-iters", "60"]], ["w16_ms64", ["--batch", "32768", "--world", "16", "--res", "32", "--max-steps", "64", "--iters", "60", "--graph-iters", "60"]], ["degenerate_w16_ground8", ["--batch", "32768", "--world", "16", "--res", "32", "--max-steps", "32", "--ground", "8", "--measure-degenerate", "--iters", "200", "--graph-iters", "200"]]]

lines = []
def say(*a):
    s = " ".join(str(x) for x in a)
    print(s, flush=True)
    lines.append(s)

say("=" * 72)
say("minecu_ref -- hand-written CUDA baseline on Kaggle")
say("=" * 72)

# ---------------------------------------------------------------- provenance
say("")
say("--- provenance ---")
say(f"  notebook assembled by minecu_ref/kaggle/make_notebook.py")
for name, b64, want in PAYLOADS:
    data = base64.b64decode(b64)
    got = hashlib.sha256(data).hexdigest()
    status = "OK" if got == want else "MISMATCH"
    say(f"  {name:14s} {len(data):8d} B  sha256 {got}  {status}")
    if got != want:
        say("")
        say("FATAL: payload hash mismatch. Refusing to measure. A timing from an")
        say("unverified source is not a timing. Re-run minecu_ref/kaggle/run.sh.")
        open(f"{OUT}/SUMMARY.txt", "w").write("\n".join(lines) + "\nHARNESS HASH MISMATCH\n")
        sys.exit(9)
    open(f"{OUT}/{name}", "wb").write(data)
    if want in EXPECT:
        say(f"                 matches the vendored copy in minecu_ref/src or bench/")

# ------------------------------------------------------------------- build
say("")
say("--- build for sm_75 ---")
NVCC = None
for p in ("/usr/local/cuda/bin/nvcc", "nvcc"):
    try:
        subprocess.run([p, "--version"], check=True, capture_output=True)
        NVCC = p
        break
    except Exception:
        continue
if NVCC is None:
    say("FATAL: no nvcc on this runner. The T4 image is supposed to have one.")
    open(f"{OUT}/SUMMARY.txt", "w").write("\n".join(lines) + "\nNO NVCC\n")
    sys.exit(10)
ver = subprocess.run([NVCC, "--version"], capture_output=True, text=True).stdout
say("  " + ver.strip().splitlines()[-2].strip())
say("  " + ver.strip().splitlines()[-1].strip())

t0 = time.time()
# -arch=sm_75 is explicit and is the whole point: the T4 is sm_75 and this run
# exists to produce a number for that architecture, not for whatever nvcc
# defaults to on an unknown card.
build = subprocess.run(
    [NVCC, "-O3", "--use_fast_math", "-std=c++17", "-arch=sm_75",
     "-diag-suppress", "177",
     f"{OUT}/kernels.cu", f"{OUT}/bench.cu", "-o", f"{OUT}/minecu_bench_sm_75"],
    capture_output=True, text=True)
say(f"  nvcc -arch=sm_75 rc={build.returncode} in {time.time()-t0:.1f}s")
if build.stderr.strip():
    for ln in build.stderr.strip().splitlines()[-25:]:
        say("  | " + ln)
if build.returncode != 0:
    open(f"{OUT}/SUMMARY.txt", "w").write("\n".join(lines) + "\nNVCC FAILED\n")
    sys.exit(11)
say(f"  binary {os.path.getsize(f'{OUT}/minecu_bench_sm_75')} B at {OUT}/minecu_bench_sm_75")

# The instruction census, so the run carries its own evidence that the binary
# is sm_75 and carries no sm_80+ instructions.
say("")
say("--- SASS census (proves the binary really is sm_75) ---")
sass = subprocess.run(["/usr/local/cuda/bin/cuobjdump", "-sass", f"{OUT}/minecu_bench_sm_75"],
                      capture_output=True, text=True)
open(f"{OUT}/kernels_sm75.sass", "w").write(sass.stdout)
gates = ["HMMA", "HGMMA", "LDGSTS", "REDUX", "ELECT", "BAR.RED", "F2FP"]
counts = {g: sass.stdout.count(g) for g in gates}
say("  arch line: " + next((l.strip() for l in sass.stdout.splitlines()
                            if l.strip().startswith("arch =")), "<none>"))
say("  " + "  ".join(f"{g}={counts[g]}" for g in gates))
say("  verdict: " + ("no sm_75-illegal instructions" if all(v == 0 for v in counts.values())
                     else "ILLEGAL INSTRUCTIONS PRESENT"))
say(f"  STG.E.U8.SYS (uint8 obs store) {sass.stdout.count('STG.E.U8.SYS')}, "
    f"LDG.E.S8.CONSTANT.SYS (__ldg voxel read) {sass.stdout.count('LDG.E.S8.CONSTANT.SYS')}")

# ----------------------------------------------------------------- battery
say("")
say("--- battery ---")
say(f"  {len(BATTERY)} entries, wall budget {BUDGET_S}s")
t_start = time.time()
summary_rows = []
dda_rows = []
skipped = []
for label, args in BATTERY:
    left = BUDGET_S - (time.time() - t_start)
    if left <= 30:
        skipped.append((label, "out of budget"))
        say(f"  SKIP {label}: out of budget")
        continue
    say("")
    say(f"  === {label}: {' '.join(args)} ===")
    t = time.time()
    try:
        p = subprocess.run([f"{OUT}/minecu_bench_sm_75"] + args,
                           capture_output=True, text=True, timeout=max(30, left))
        rc, body = p.returncode, p.stdout + ("\n[stderr]\n" + p.stderr if p.stderr else "")
    except subprocess.TimeoutExpired:
        rc, body = -9, "TIMEOUT"
    dt = time.time() - t
    open(f"{OUT}/log_{label}.txt", "w").write(body)
    for ln in body.rstrip().splitlines():
        say("  " + ln)
    say(f"  --- {label}: rc={rc} in {dt:.1f}s -> log_{label}.txt")
    if rc != 0:
        say(f"  !!! {label} exited {rc}. The harness itself is suspect here, not the kernel.")
    # pull the headline numbers for the summary
    for ln in body.splitlines():
        if "launch_step (fp32 obs" in ln or "launch_step_uint8_prebasis (" in ln \
           or "launch_step_..._prebasis_full" in ln:
            parts = ln.split()
            if len(parts) >= 8:
                summary_rows.append((label, parts[1] + " " + parts[2], parts[-1]))
    # The DDA step count is what says whether a timing is a voxel-render
    # measurement at all, so it goes in the summary next to the throughput
    # rather than only in the per-entry log. Pulled from the selftest line,
    # which reads "mean <N> DDA steps/pixel over <P> pixels".
    for ln in body.splitlines():
        if "DDA steps/pixel over" in ln:
            mm = re.search(r"mean ([\d.]+) DDA steps/pixel over (\d+) pixels", ln)
            if mm:
                dda_rows.append((label, mm.group(1), mm.group(2)))
                break
    if rc == 0 and "selftest: PASS" not in body and "selftest" in " ".join(args):
        summary_rows.append((label, "SELFTEST", "FAIL"))

say("")
say("--- summary: throughput ---")
say(f"{'entry':28s} {'path':44s} {'M env-steps/s'}")
for label, path, v in summary_rows:
    say(f"{label:28s} {path:44s} {v:>8s}")
for label, why in skipped:
    say(f"{label:28s} {'-':44s} {'SKIP':>8s}  ({why})")

say("")
say("--- summary: DDA workload actually timed ---")
say("  A throughput number means nothing without the workload behind it. These are")
say("  the mean DDA steps per pixel the timed render kernel executed, measured on")
say("  the device by replaying the vendored DDA over the same voxels and basis.")
say(f"{'entry':28s} {'mean DDA steps/pixel':>22s} {'pixels':>12s}")
for label, mean, pix in dda_rows:
    say(f"{label:28s} {mean:>22s} {pix:>12s}")
say("")
say("  headline_w16 is the number to use. degenerate_w16_ground8 is the SAME shape")
say("  with ground=8, the configuration the previous run measured; the agent's eye")
say("  was three blocks inside its own surface, every ray hit on its first voxel")
say("  test, and the render kernel degenerated into an obs store. It is in this")
say("  run so the before/after is measured on the same card, not quoted.")

say("")
say("Budget: %d/%d s used." % (time.time() - t_start, BUDGET_S))
say("")
say("READING THIS OUTPUT")
say("  * The three selftest_* entries gate everything. If any says FAIL, the")
say("    numbers below it are not trustworthy and the harness is the suspect.")
say("  * Each entry also gates on a DDA step count. An entry that reports a")
say("    throughput but a mean of ~1.000 steps/pixel is NOT a voxel-render")
say("    measurement: the render exited on its first voxel test every pixel.")
say("    bench.cu refuses to report timings for such a config unless")
say("    --measure-degenerate is passed, which is used exactly once here, for the")
say("    entry that exists to document the pre-fix baseline.")
say("  * env-steps/s is batch / median_us. A whole-step number, not a per-kernel one.")
say("  * The measured memory floors in each log are the denominators. %-of-floor")
say("    figures only mean something against a floor measured on the same card.")
say("  * The author's 49.6M is an RTX 3090 claim. This is a T4. They are not")
say("    comparable numbers and must not be differenced directly.")
summary = "\n".join(lines)
open(f"{OUT}/SUMMARY.txt", "w").write(summary + "\n")
say("wrote /kaggle/working/SUMMARY.txt")
